# Week 63, Day 2: EM as coordinate ascent on the ELBO

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w063/d2.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: ELBO and log-likelihood through EM

Write elbo(x, R, pi, mu, sd), the ELBO of a one-dimensional Gaussian mixture for responsibilities R: the sum over points and components of R times (log pi + log normal density - log R). Run 30 EM iterations, recording the ELBO after each E step and after each M step, and the log-likelihood.

In [ ]:
import numpy as np
from scipy.stats import norm
g0 = np.random.default_rng(632)
x = np.r_[g0.normal(-2.5, 0.7, 150), g0.normal(1.5, 1.0, 250)]

def loglik(x, pi, mu, sd):
    return np.log((pi * norm.pdf(x[:, None], mu, sd)).sum(1)).sum()

def e_step(x, pi, mu, sd):
    a = pi * norm.pdf(x[:, None], mu, sd)
    return a / a.sum(1, keepdims=True)

def m_step(x, R):
    Nk = R.sum(0)
    mu = (R * x[:, None]).sum(0) / Nk
    sd = np.sqrt((R * (x[:, None] - mu) ** 2).sum(0) / Nk)
    return Nk / len(x), mu, sd

def elbo(x, R, pi, mu, sd):
    logjoint = np.log(pi) + norm.logpdf(x[:, None], mu, sd)
    return ...

pi, mu, sd = np.array([0.5, 0.5]), np.array([-0.5, 2.5]), np.array([1.5, 1.5])
elbo_after_E, elbo_after_M, ll_trace = [], [], []
for it in range(30):
    R = e_step(x, pi, mu, sd)
    elbo_after_E.append(elbo(x, R, pi, mu, sd)); ll_trace.append(loglik(x, pi, mu, sd))
    pi, mu, sd = m_step(x, R)
    elbo_after_M.append(elbo(x, R, pi, mu, sd))
for it in (0, 1, 2, 5, 29):
    print(f"iteration {it:2d}: log-likelihood {ll_trace[it]:9.3f}  ELBO after E {elbo_after_E[it]:9.3f}  after M {elbo_after_M[it]:9.3f}")
print("fitted:", np.round(pi, 3), np.round(mu, 3), np.round(sd, 3))

In [ ]:
import numpy as np
def test_touch():
    "after every E step the ELBO equals the log-likelihood"
    assert all(abs(e - l) < 1e-8 for e, l in zip(elbo_after_E, ll_trace))
def test_up():
    "every M step raises the ELBO, and the log-likelihood never decreases"
    assert all(m >= e - 1e-10 for e, m in zip(elbo_after_E, elbo_after_M))
    assert all(b >= a - 1e-10 for a, b in zip(ll_trace, ll_trace[1:]))

test_touch()
test_up()
print("All checks passed")

### Exercise: Updating a block at a time

Implement incremental EM with 10 blocks: keep the sufficient statistics (sum of R, sum of R x, sum of R x squared), and for each block subtract its old contribution, recompute its responsibilities, add the new contribution, and do an M step from the statistics. Compare the log-likelihood after each full pass with standard EM.

In [ ]:
def params_from_stats(s0, s1, s2):
    mu_ = s1 / s0
    return s0 / s0.sum(), mu_, np.sqrt(np.maximum(s2 / s0 - mu_ ** 2, 1e-6))

def stats(xb, Rb):
    return Rb.sum(0), (Rb * xb[:, None]).sum(0), (Rb * xb[:, None] ** 2).sum(0)

start = (np.array([0.5, 0.5]), np.array([-0.5, 2.5]), np.array([1.5, 1.5]))
perm = np.random.default_rng(6321).permutation(len(x))
blocks = np.array_split(perm, 10)

p_, m_, s_ = start; ll_std = []
for it in range(20):
    p_, m_, s_ = m_step(x, e_step(x, p_, m_, s_)); ll_std.append(loglik(x, p_, m_, s_))

p_, m_, s_ = start
R_all = e_step(x, p_, m_, s_)
S0, S1, S2 = stats(x, R_all)
ll_inc = []
for sweep in range(20):
    for b in blocks:
        o0, o1, o2 = stats(x[b], R_all[b])
        R_all[b] = e_step(x[b], p_, m_, s_)
        n0, n1, n2 = stats(x[b], R_all[b])
        S0, S1, S2 = ...
        p_, m_, s_ = params_from_stats(S0, S1, S2)
    ll_inc.append(loglik(x, p_, m_, s_))
print("log-likelihood after passes 1, 2, 3, 20")
print("  standard EM:   ", np.round([ll_std[i] for i in (0, 1, 2, 19)], 3))
print("  incremental EM:", np.round([ll_inc[i] for i in (0, 1, 2, 19)], 3))

In [ ]:
import numpy as np
def test_incremental():
    "after two passes, incremental EM has a higher log-likelihood than two iterations of standard EM"
    assert ll_inc[1] > ll_std[1]
def test_converge():
    "after 20 passes both reach the same log-likelihood to within 0.01"
    assert abs(ll_inc[-1] - ll_std[-1]) < 0.01

test_incremental()
test_converge()
print("All checks passed")

### Exercise: One gradient step in place of the M step

Replace the M step for the means by one gradient-ascent step on the expected complete-data log-likelihood, with step size 0.1 times sd squared divided by N_k (the weights and standard deviations keep their exact updates). Run 200 iterations and record the log-likelihood.

In [ ]:
p_, m_, s_ = start
ll_gem = []
for it in range(200):
    R = e_step(x, p_, m_, s_)
    Nk = R.sum(0)
    grad_mu = (R * (x[:, None] - m_)).sum(0) / s_ ** 2          # derivative of E_q[log p] with respect to each mean
    m_ = ...
    p_ = Nk / len(x)
    s_ = np.sqrt((R * (x[:, None] - m_) ** 2).sum(0) / Nk)
    ll_gem.append(loglik(x, p_, m_, s_))
print("generalised EM, log-likelihood at iterations 1, 10, 50, 200:", np.round([ll_gem[i] for i in (0, 9, 49, 199)], 3))
print(f"full EM after 20 iterations: {ll_std[-1]:.3f}")

In [ ]:
import numpy as np
def test_gem():
    "with a small gradient step for the means, the log-likelihood never decreases and ends within 1 of full EM"
    assert all(b >= a - 1e-8 for a, b in zip(ll_gem, ll_gem[1:])) and ll_gem[-1] > ll_std[-1] - 1

test_gem()
print("All checks passed")